In [1]:
!git clone https://github.com/Tahoni01/Continual-hate-speech-detection.git
%cd Continual-hate-speech-detection
!ls

Cloning into 'Continual-hate-speech-detection'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 96 (delta 35), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (96/96), 180.19 KiB | 4.29 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/Continual-hate-speech-detection
dataset  Main.ipynb  models  old_main.ipynb  README.md	requirements.txt  src


In [3]:
import torch

from dataset.df_loader import (
    getdf_davidson,
    getdf_hatexplain
)

from dataset.stream_generator import (
    create_continual_stream,
    online_stream
)

from transformers import (
    AutoTokenizer,
    AutoConfig
)

from models.model_builder import CustomClassifier

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)

DEVICE: cpu


In [5]:
print("Loading datasets...")

df_dv = getdf_davidson()
df_hx = getdf_hatexplain()

print("Davidson:", df_dv.shape)
print("HateXplain:", df_hx.shape)

Loading datasets...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.63M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24783 [00:00<?, ? examples/s]

                                                text      label    source
0  rt as a woman you shouldnt complain about clea...     normal  davidson
1  rt boy dats coldtyga dwn bad for cuffin dat ho...  offensive  davidson
2  rt dawg rt you ever fuck a bitch and she start...  offensive  davidson
3                          rt she look like a tranny  offensive  davidson
4  rt the shit you hear about me might be true or...  offensive  davidson
                                                text       label      source
0  i dont think im getting my baby them white 9 h...      normal  hatexplain
1  we cannot continue calling ourselves feminists...      normal  hatexplain
2                      nawt yall niggers ignoring me      normal  hatexplain
3  user i am bit confused coz chinese ppl can not...  hatespeech  hatexplain
4  this bitch in whataburger eating a burger with...  hatespeech  hatexplain
Davidson: (24275, 3)
HateXplain: (20144, 3)


In [6]:
full_stream = create_continual_stream(
    df_list=[df_dv, df_hx],
    batch_size=32
)

In [7]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
num_labels = 2

config = AutoConfig.from_pretrained(
    "roberta-base",
    num_labels=num_labels
)

In [19]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="peft")
!pip install --upgrade torchao

model = CustomClassifier(
    model_name="roberta-base",
    config=config,
    class_weights=None,
    use_lora=True
).to(device)

print("Model loaded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 60.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded


In [ ]:
print("Starting stream simulation...\n")

stream_iterator = online_stream(full_stream)

for step, batch in enumerate(stream_iterator):

    print("=" * 50)
    print(f"STREAM STEP {step}")
    print("=" * 50)

    print(batch.head())

    print("\nBatch size:", len(batch))

    print("\nLabel distribution:")
    print(batch["label"].value_counts())

    # ---------------------------------
    # FUTURE:
    # train_step(batch)
    # ---------------------------------

    if step == 50:
        break